# Chapter 13 &mdash; Halting, Acceptance, and Why a TM Need Not Read Its Input

**Concept 7 of the Chapter 13 decomposition:** *Halting, Acceptance, and Why a TM Need Not Read Its Input*

A TM halts when stuck; accepts if stuck in $F$ &mdash; and it need never read a single input symbol.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter13/Concept-Halting-And-Acceptance/Concept-Halting-And-Acceptance.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_PDA        import *
from jove.Def_TM         import *
from jove.AnimateTM      import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateTM as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateTM, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


Three facts that trip people up:

* a TM **halts when no transition applies** &mdash; when it is *stuck*. There is no
  explicit "halt" instruction;
* it **accepts** if it halts in a final state, and **rejects** if it halts elsewhere;
* it may also **run forever**, which is neither.

And a consequence that feels wrong the first time: **a TM need not read its input at
all.** Nothing forces the head to move, and a machine that immediately halts in $F$
accepts *every* string, including ones it never looked at.

This is why an accepting state in Jove must have **no outgoing transitions**. Leave one
in and the machine runs on past the moment you thought it had finished.

## 2. Definitions

### Three machines: accept, reject, diverge

In [ ]:
# NOTE: md2mc does not expand '|' in a TM's READ field -- it silently
# keeps only the last alternative.  Write one line per symbol.
Acc = md2mc('''TM
I : 0 ; . , S -> F      !! F has no outgoing edges: halt and accept
I : 1 ; . , S -> F
I : . ; . , S -> F
''')
Rej = md2mc('''TM
I : 0 ; 0 , R -> D              !! D is not final and has no edges: halt, reject
''')
Div = md2mc('''TM
I : 0 ; 0 , R -> I      !! never stuck, so never halts
I : 1 ; 1 , R -> I
I : . ; . , R -> I
''')

# --- thin wrappers over Jove's TM runner --------------------------------
# run_tm(T, tape, fuel) returns (truncated-paths, haltList).  A TM HALTS
# when no transition applies, and ACCEPTS if it halts in a final state.
# So an accepting state must have NO outgoing transitions, or the machine
# will run on past it.
def tm_accepts(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return any(cfg[0] in T["F"] for cfg, _ in halts)

def tm_halts(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return len(halts) > 0

def tm_tape(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return [cfg[2].rstrip('.') for cfg, _ in halts]

<!-- nav-strip -->

---

&larr;&nbsp;[Ch13&nbsp;6.&nbsp;The Formal Turing Machine $(Q,\Sigma,\Gamma,\Delta,q_0,B,F)$](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter13/Concept-Formal-TM/Concept-Formal-TM.ipynb) &nbsp;&middot;&nbsp; [**Chapter 13** index](https://github.com/ganeshutah/Jove/blob/master/Chapter13/README.md) &nbsp;&middot;&nbsp; [Ch13&nbsp;8.&nbsp;Simple TM Examples: Bit Flipper and "Contains 101"](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter13/Concept-Simple-TM-Examples/Concept-Simple-TM-Examples.ipynb)&nbsp;&rarr;

---

## 3. Tests

**Accept:** halt in a final state.

In [ ]:
for t in ['0', '1', '0000']:
    print("  %-7r halts %-6s accepts %s" % (t, tm_halts(Acc, t), tm_accepts(Acc, t)))
assert tm_accepts(Acc, '0') and tm_accepts(Acc, '1111')

**Reject:** halt somewhere else.

In [ ]:
print("  '0' halts?", tm_halts(Rej, '0'), "  accepts?", tm_accepts(Rej, '0'))
assert tm_halts(Rej, '0') and not tm_accepts(Rej, '0')
print("\nD is a dead end but not final, so this is a rejection, not a loop.")

**Diverge:** never stuck, so never halts.

In [ ]:
for fuel in [10, 100, 500]:
    print("  fuel %3d : halts? %s" % (fuel, tm_halts(Div, '0', fuel=fuel)))
assert not tm_halts(Div, '0', fuel=500)
print("\nThis is the third outcome, and it is the one that makes halting undecidable.")

**A TM need not read its input.** `Acc` accepts everything, looking at one cell.

In [ ]:
from itertools import product
strs = [''.join(p) for k in range(1, 6) for p in product('01', repeat=k)]
assert all(tm_accepts(Acc, s) for s in strs)
print("accepts all %d strings tried, in one move each" % len(strs))
print("\nL(Acc) = Sigma*, and the head never moved.")

**The trap:** a final state with an outgoing transition is not a halt.

In [ ]:
Oops = md2mc('''TM
I : 0 ; 0 , R -> F
F : 0 ; 0 , R -> F      !! BUG: F keeps going
F : 1 ; 1 , R -> F
F : . ; . , R -> F
''')
print("  halts on '0' ? ", tm_halts(Oops, '0', fuel=200))
print("  accepts '0'  ? ", tm_accepts(Oops, '0', fuel=200))
assert not tm_halts(Oops, '0', fuel=200)
print("\nIt reached F and then ran off to the right forever.  Never halted,")
print("so never accepted.  Give your final states NO outgoing transitions.")

## 4. Animation

A machine that accepts without reading anything.

In [ ]:
from jove.AnimateTM import *
AnimateTM(Acc, FuseEdges=True)

## 5. Exercises


1. Write a TM that rejects every string. How few transitions do you need?
2. Is "halts" the same as "decides"? What is the difference?
3. Why can a DFA never diverge, but a TM can?

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
from jove.Nav import nav, load_here
nav(here='Chapter13/Concept-Halting-And-Acceptance')